In [1]:
!pip -q install groq datasets sentence-transformers rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.8 MB/s eta 0:00:00


In [2]:
import os, json, re, time, random, unicodedata, itertools
import numpy as np, pandas as pd
from google.colab import userdata, drive

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- OpenRouter (primary transport) ----
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Add OPENROUTER_API_KEY to Colab Secrets"

# ---- Groq (optional fallback) ----
GROQ_KEYS = []
for i in range(1, 10):
    try:
        k = userdata.get(f"GROQ_API_KEY_{i}")
        if k: GROQ_KEYS.append(k)
    except Exception:
        pass
print(f"OpenRouter: ready | Groq fallback keys: {len(GROQ_KEYS)}")

# ---- HF token (uppercase - required, see Legal session) ----
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # prevents download stalls

# ---- GitHub ----
PUSH_TOKEN = userdata.get("GIT_PAT_V")
REPO_OWNER, REPO_NAME = "veenulearns-lab", "RAGBench-Capstone-Batch26"

drive.mount("/content/drive", force_remount=False)
SAVE_PATH = "/content/drive/MyDrive/RAGBench_Results/NEW_CS_Aug"
os.makedirs(SAVE_PATH, exist_ok=True)
print("save path:", SAVE_PATH)

OpenRouter: ready | Groq fallback keys: 5
Mounted at /content/drive
save path: /content/drive/MyDrive/RAGBench_Results/NEW_CS_Aug


In [8]:
import os

# Run just one experiment
os.environ["EXP_ONLY"] = "CS-064"
# os.environ.pop("EXP_ONLY", None)      # Remove any previous single-experiment setting
os.environ["FORCE_RERUN"] = "1"
# os.environ.pop("FORCE_RERUN", None)

# Export Groq keys to environment for standalone scripts
if GROQ_KEYS:
    os.environ["GROQ_API_KEY"] = GROQ_KEYS[0]
    os.environ["GROQ_API_KEYS"] = ",".join(GROQ_KEYS)

# Export OpenRouter too (if you use it elsewhere)
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [9]:
import os, pathlib

# --- paths: LOCAL must NOT be on Drive, or you write to Drive twice ---
os.environ["SAVE_PATH"]  = "/content/cs_local"                                  # scratch
os.environ["LOCAL_ROOT"] = "/content/cs_local"                                  # fast local leg
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/RAGBench_Results/NEW_CS_Aug" # Drive leg
pathlib.Path("/content/cs_local").mkdir(parents=True, exist_ok=True)

# --- secrets: subprocess can't read Colab userdata, so export them ---
os.environ["GIT_PAT_V"]  = PUSH_TOKEN
os.environ["REPO_ROOT"]  = "/content/RAGBench-Capstone-Batch26"
os.environ["ENABLE_GIT"] = "1" if os.path.isdir("/content/RAGBench-Capstone-Batch26/.git") else "0"

# --- cadence + run controls ---
os.environ["CHECKPOINT_EVERY"]      = "5"   # local+Drive save every 5 examples
os.environ["GIT_PUSH_EVERY_N_CKPT"] = "5"   # git push every 5 checkpoints (25 examples)
os.environ["N_EXAMPLES"]    = "200"           # 1 smoke -> 25 iterate -> 200 final
os.environ["DENSE_ONLY"]    = "0"
os.environ["GROQ_COOLDOWN"] = "1.0"
os.environ["GUARD_PAUSE"]   = "3"           # set to 3 for the N=200 run

In [10]:
!pip install -q openai

In [14]:
%%javascript
function KeepAlive() {
    document.querySelector("#toggle-header-button").click();
    setTimeout(KeepAlive, 60000);
}
KeepAlive();


<IPython.core.display.Javascript object>

In [13]:
!python /content/customer_support_rag_standalone.py customer_support


Streaming output truncated to the last 5000 lines.





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 





 






In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

False
CPU


In [5]:
# ===== CS report export — paste-and-run cell =====
import os
import pandas as pd

BASE           = "/content/drive/MyDrive/RAGBench_Results/NEW_CS_Aug/customer_support"
N_FILTER       = 25      # None = all sample sizes
MIN_VALID_FRAC = 0.80    # judge coverage below this = not report-grade
SORT_BY        = "adherence_auroc"
SHOW_EXP_ID    = True

COLUMNS = [
    ("Judge Model", "judge_model"), ("Embedding Model", "embedder"),
    ("Chunking Strategy", "chunk_config"), ("Generator LLM", "gen_model"),
    ("Retrieval Method", "retrieval"), ("Re-ranking Method", "rerank"),
    ("Context Ordering", "context_order"),
    ("TRACe Context Relevance \u2191", "context_relevance"),
    ("TRACe Context Utilization \u2191", "context_utilization"),
    ("TRACe Completeness \u2191", "completeness"),
    ("TRACe Adherence \u2191", "adherence"),
    ("Valid_Judge_response", "_valid"),
    ("Context Relevance RMSE \u2193", "context_relevance_rmse"),
    ("Context Utilization RMSE \u2193", "context_utilization_rmse"),
    ("Completeness RMSE \u2193", "completeness_rmse"),
    ("Adherence AUROC \u2191", "adherence_auroc"),
]

df = pd.read_csv(f"{BASE}/results_customer_support.csv")
print(f"master rows: {len(df)}")

if N_FILTER is not None:
    df = df[df["n_examples"].astype(str) == str(N_FILTER)]

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
n0 = len(df)
df = df.sort_values("timestamp").drop_duplicates("exp_id", keep="last").reset_index(drop=True)
if n0 != len(df):
    print(f"de-duplicated {n0} -> {len(df)} (newest row per exp_id)")

df["judge_used"] = pd.to_numeric(df["judge_used"], errors="coerce").fillna(0.0)
df["n_examples"] = pd.to_numeric(df["n_examples"], errors="coerce").fillna(0).astype(int)
df["_valid"] = ((df["judge_used"] * df["n_examples"]).round().astype(int).astype(str)
                + "/" + df["n_examples"].astype(str))
df["Status"] = ["report-grade" if u >= MIN_VALID_FRAC else "LOW JUDGE COVERAGE"
                for u in df["judge_used"]]
df["_rank"]  = (df["Status"] != "report-grade").astype(int)
df["_s"]     = pd.to_numeric(df.get(SORT_BY), errors="coerce")
df = df.sort_values(["_rank", "_s"], ascending=[True, False], na_position="last")

out = pd.DataFrame()
if SHOW_EXP_ID:
    out["Experiment ID"] = df["exp_id"]
for label, src in COLUMNS:
    out[label] = df[src] if src in df.columns else ""
out["Status"] = df["Status"]

ok, bad = out[out.Status == "report-grade"], out[out.Status != "report-grade"]
for frame, name in ((out, "ALL"), (ok.drop(columns="Status"), ""),
                    (bad, "EXCLUDED")):
    suffix = f"_{name}" if name else ""
    frame.to_csv(f"{BASE}/report_customer_support{suffix}.csv",
                 index=False, encoding="utf-8-sig")   # BOM keeps ↑ ↓ in Excel

print(f"report-grade: {len(ok)} | low coverage: {len(bad)} | total: {len(out)}")
if len(bad):
    print("excluded: " + ", ".join(f"{e}({v})" for e, v in
                                   zip(bad["Experiment ID"], bad["Valid_Judge_response"])))

with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.width", 300):
    print("\n" + out.to_string(index=False))

master rows: 147
report-grade: 77 | low coverage: 19 | total: 96
excluded: CS-041(14/25), CS-095(18/25), CS-048(12/25), CS-042(14/25), CS-047(11/25), CS-040(14/25), CS-033(12/25), CS-096(17/25), CS-046(11/25), CS-044(2/25), CS-038(3/25), CS-094(16/25), CS-043(2/25), CS-045(2/25), CS-034(9/25), CS-037(3/25), CS-039(3/25), CS-036(9/25), CS-035(8/25)

Experiment ID         Judge Model   Embedding Model Chunking Strategy           Generator LLM Retrieval Method Re-ranking Method Context Ordering  TRACe Context Relevance ↑  TRACe Context Utilization ↑  TRACe Completeness ↑  TRACe Adherence ↑ Valid_Judge_response  Context Relevance RMSE ↓  Context Utilization RMSE ↓  Completeness RMSE ↓  Adherence AUROC ↑             Status
       CS-064 openai/gpt-oss-120b  all-MiniLM-L6-v2       sliding_5o2 llama-3.3-70b-versatile           hybrid     cross_encoder          forward                     0.2767                       0.1839                0.5571             0.5969                24/25         